# Selected Feature Extraction (All Datasets)


# Parameterised entropy / thermodynamic / (EyeFeatures)


In [ ]:
import warnings

from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

from src.features.extractor import Extractor

from src.utils.dataset_utils import (
    find_all_datasets,
    load_and_preprocess_dataset,
)

DATASETS_DIR = Path("data")
OUTPUT_DIR = Path("new_features")
OUTPUT_DIR.mkdir(exist_ok=True)

OUT_SUBDIR = OUTPUT_DIR
OUT_SUBDIR.mkdir(exist_ok=True)


In [ ]:
from src.features.new_features import (
    StateDistributionEntropy,
    WeightedStateDistributionEntropy,
    EdgeDistributionEntropy,
    ThermodynamicObservables,
    ThermodynamicObservablesExtended,
    ThermodynamicTransitionDivergence,
)



In [ ]:
def consolidate_and_save_features(all_results: dict, output_dir: Path = OUT_SUBDIR):
    """Consolidate all extracted features by dataset and save one CSV per dataset."""
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    dataset_features = defaultdict(list)

    print(f"\n{'='*80}")
    print("CONSOLIDATING FEATURES BY DATASET")
    print(f"{'='*80}\n")

    for feature_family, family_results in all_results.items():
        for config_name, dataset_results in family_results.items():
            for dataset_name, result in dataset_results.items():
                if result.get("status") != "success":
                    continue
                df = result.get("features_df")
                if df is None or len(df) == 0:
                    continue
                dataset_features[dataset_name].append(df)

    print(f"Found features for {len(dataset_features)} datasets\n")

    consolidated_files: dict[str, Path] = {}

    for dataset_name, dfs in dataset_features.items():
        print(f"Consolidating {dataset_name} ({len(dfs)} feature blocks)...")

        try:
            if not dfs:
                continue

            index_col = dfs[0].columns[0]
            merged = dfs[0].copy()

            for i, df in enumerate(dfs[1:], 1):
                if index_col not in df.columns:
                    print(f"  ⚠️ missing index column '{index_col}' in block {i}, skipping")
                    continue
                merged = merged.merge(df, on=index_col, how="outer", suffixes=("", f"_dup_{i}"))

            seen = set([index_col])
            drop_cols = []
            rename_map = {}

            for col in merged.columns:
                if col == index_col:
                    continue
                base = col.rsplit("_dup_", 1)[0] if "_dup_" in col else col
                if base in seen:
                    drop_cols.append(col)
                else:
                    seen.add(base)
                    if col != base:
                        rename_map[col] = base

            if rename_map:
                merged = merged.rename(columns=rename_map)
            if drop_cols:
                merged = merged.drop(columns=drop_cols)

            out_file = output_dir / f"{dataset_name}_new.csv"
            merged.to_csv(out_file, index=False)
            consolidated_files[dataset_name] = out_file
            print(f"  ✅ Saved {out_file.name}: {merged.shape[0]} rows × {merged.shape[1]} cols")

        except Exception as e:
            print(f"  ❌ Failed to consolidate {dataset_name}: {e}")

    return consolidated_files

In [ ]:
all_datasets = find_all_datasets(DATASETS_DIR)
datasets_to_process = all_datasets["fixation"] + all_datasets["unknown"]

print(f"Found {len(datasets_to_process)} fixation datasets to process")
print(f"  - Explicit fixation datasets: {len(all_datasets['fixation'])}")
print(f"  - Unknown datasets (treated as fixations): {len(all_datasets['unknown'])}")

print("\nLoading all datasets...")
loaded_datasets: dict[str, dict] = {}
for dataset_path in datasets_to_process:
    dataset_name = dataset_path.stem
    try:
        df, col_info, _ = load_and_preprocess_dataset(dataset_path)
        if df is None or len(df) == 0:
            continue

        if "duration" in df.columns and col_info.get("duration_col") is None:
            col_info["duration_col"] = "duration"

        group_cols = col_info.get("group_cols") or []
        if isinstance(group_cols, str):
            group_cols = [group_cols]

        if len(group_cols) > 0:
            group_sizes = df.groupby(group_cols).size()
            if group_sizes.max() <= 5:
                print(f"⚠️  Skipping {dataset_name}: insufficient data (max scanpath size: {group_sizes.max()})")
                continue

        loaded_datasets[dataset_name] = {"df": df, "col_info": col_info, "path": dataset_path}
        print(f"✅ Loaded {dataset_name}: {len(df):,} rows")
    except Exception as e:
        print(f"❌ Error loading {dataset_name}: {e}")

print(f"\nSuccessfully loaded {len(loaded_datasets)} datasets\n")


In [ ]:
from src.utils.feature_extraction_utils import get_split_info_paths_for_dataset

SPLITS_DIR = Path("extensive_features") / "splits"

try:
    from src.utils.path_pk_config import PATH_PK_PER_LABEL
except ImportError:
    PATH_PK_PER_LABEL = {}
    print("path_pk_config not found; path_pk will fall back to choose_path_pk(col_info) per dataset")

In [ ]:
def choose_path_pk(col_info: dict) -> list[str]:
    """Fallback when path_pk is not taken from PATH_PK_PER_LABEL (no splits or no config).

    - pk = scanpath id columns (group_cols)
    - path_pk = columns for grouping scanpaths into 'prototype families'.
      We default to group_cols[:-1] when not using label-wise PATH_PK_PER_LABEL.
    """
    group_cols = col_info.get("group_cols") or []
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    if len(group_cols) >= 2:
        return list(group_cols[:-1])
    return list(group_cols)


In [ ]:
def extract_feature_with_configs(feature_class, feature_name: str, configs: list[dict]):
    """Extract a feature with multiple hyperparameter configurations.

    This function:
    - instantiates the feature class for each config
    - runs fit_transform per dataset
    - reports NaNs and small variance preview
    - stores the resulting df in `all_results`
    """
    import inspect

    target_output_dir = globals().get("CURRENT_OUTPUT_DIR")
    if target_output_dir is not None:
        target_output_dir = Path(target_output_dir)

    results = {}

    for i, config in enumerate(configs):
        config_name = f"{feature_name}_{i+1}"
        print(f"\n🔧 {config_name}")
        print(f"   Config: {config}")

        dataset_results = {}

        for dataset_name, dataset_data in loaded_datasets.items():
            if target_output_dir is not None and (target_output_dir / f"{dataset_name}_new.csv").exists():
                print(f"    ⏭️ {dataset_name}: output exists, skipping")
                dataset_results[dataset_name] = {"status": "skipped", "reason": "output_exists"}
                continue

            df = dataset_data["df"]
            col_info = dataset_data["col_info"]

            x_col = col_info.get("x_col")
            y_col = col_info.get("y_col")
            t_col = col_info.get("t_col")
            dur_col = col_info.get("duration_col")
            aoi_col = col_info.get("aoi_col")
            group_cols = col_info.get("group_cols") or []
            if isinstance(group_cols, str):
                group_cols = [group_cols]

            feature_kwargs = {"pk": group_cols if group_cols else None, "return_df": True}
            feature_kwargs.update(config)

            try:
                sig = inspect.signature(feature_class.__init__)
                accepted_params = set(sig.parameters.keys())
            except (TypeError, ValueError):
                accepted_params = set()

            def _maybe_set_kw(name: str, value):
                if value is None:
                    return
                if not accepted_params or name in accepted_params:
                    feature_kwargs.setdefault(name, value)

            _maybe_set_kw("x", x_col)
            _maybe_set_kw("y", y_col)
            _maybe_set_kw("t", t_col)
            _maybe_set_kw("duration", dur_col)

            try:
                if feature_kwargs.get("weights") in ("duration", "dur", "fix_dur"):
                    if dur_col is None:
                        raise ValueError("duration_col missing but weights='duration' requested")
                    feature_kwargs["weights"] = dur_col

                if feature_kwargs.get("joint", "").startswith("state_duration") and dur_col is None:
                    raise ValueError("duration_col missing but joint='state_duration' requested")

                if feature_kwargs.get("energy") in ("speed",) and t_col is None:
                    raise ValueError("t_col missing but energy='speed' requested")
                if feature_kwargs.get("energy") in ("duration_inv",) and dur_col is None:
                    raise ValueError("duration_col missing but energy='duration_inv' requested")

                uses_path_pk = "path_pk" in feature_kwargs and (feature_kwargs["path_pk"] is None or len(feature_kwargs.get("path_pk") or []) == 0)
                if uses_path_pk and PATH_PK_PER_LABEL and SPLITS_DIR.exists():
                    split_paths = get_split_info_paths_for_dataset(SPLITS_DIR, dataset_name)
                    if split_paths:
                        for split_path in split_paths:
                            split_id = split_path.stem.replace("_split_info", "")
                            path_pk = PATH_PK_PER_LABEL.get(split_id, list(group_cols))
                            missing = [c for c in path_pk if c not in df.columns]
                            if missing:
                                print(f"    ⚠️ {split_id}: path_pk columns {missing} missing; skip")
                                dataset_results[split_id] = {"status": "error", "error": f"Missing path_pk: {missing}"}
                                continue
                            kw = {**feature_kwargs, "path_pk": path_pk}
                            try:
                                feat = feature_class(**kw)
                                features_df = feat.fit_transform(df)
                                if hasattr(features_df, "index") and getattr(features_df.index, "nlevels", 0) > 0:
                                    features_df = features_df.reset_index()
                                feature_cols = [c for c in features_df.select_dtypes(include=[np.number]).columns if c != "index" and c not in group_cols]
                                n_rows = len(features_df)
                                nan_total = int(features_df[feature_cols].isna().sum().sum()) if feature_cols else 0
                                nan_pct = 100.0 * nan_total / max(n_rows * max(len(feature_cols), 1), 1)
                                variances = {c: float(pd.to_numeric(features_df[c], errors="coerce").var()) for c in (feature_cols[:50]) if pd.notna(pd.to_numeric(features_df[c], errors="coerce").var())}
                                print(f"    📊 {split_id}: {n_rows} rows, {nan_total} NaNs ({nan_pct:.2f}%)")
                                if variances:
                                    var_str = ", ".join([f"{k}={v:.6g}" for k, v in list(variances.items())[:3]])
                                    print(f"       Variance preview: {var_str}")
                                dataset_results[split_id] = {"status": "success", "features_df": features_df}
                            except Exception as e:
                                print(f"    ❌ {split_id}: {e}")
                                dataset_results[split_id] = {"status": "error", "error": str(e)}
                        continue
                if uses_path_pk:
                    feature_kwargs["path_pk"] = choose_path_pk(col_info)

                feat = feature_class(**feature_kwargs)
                features_df = feat.fit_transform(df)

                if hasattr(features_df, "index") and getattr(features_df.index, "nlevels", 0) > 0:
                    features_df = features_df.reset_index()

                feature_cols = [c for c in features_df.select_dtypes(include=[np.number]).columns]
                if "index" in feature_cols:
                    feature_cols.remove("index")
                for gc in group_cols:
                    if gc in feature_cols:
                        feature_cols.remove(gc)

                n_rows = len(features_df)
                nan_total = int(features_df[feature_cols].isna().sum().sum()) if feature_cols else 0
                denom = max(n_rows * max(len(feature_cols), 1), 1)
                nan_pct = 100.0 * nan_total / denom

                variances = {}
                for col in feature_cols[:50]:
                    v = pd.to_numeric(features_df[col], errors="coerce").var()
                    if pd.notna(v):
                        variances[col] = float(v)

                print(
                    f"    📊 {dataset_name}: {n_rows} rows, {nan_total} NaNs "
                    f"({nan_pct:.2f}% of numeric values)"
                )
                if variances:
                    var_items = list(variances.items())[:3]
                    var_str = ", ".join([f"{k}={v:.6g}" for k, v in var_items])
                    more = "" if len(variances) <= 3 else f" ... ({len(variances)-3} more)"
                    print(f"       Variance preview: {var_str}{more}")

                dataset_results[dataset_name] = {"status": "success", "features_df": features_df}

            except Exception as e:
                print(f"    ❌ {dataset_name}: {e}")
                dataset_results[dataset_name] = {"status": "error", "error": str(e)}

        results[config_name] = dataset_results

    return results


In [ ]:
all_results: dict[str, dict] = {}

_GRID_SIZES_EXTRA = (2, 4, 6, 8)

def _add_grid_variants(configs: list[dict]) -> list[dict]:
    """Append variants with grid_size in (2,4,6,8) for each config that has grid_size."""
    extra = [dict(c, grid_size=g) for c in configs for g in _GRID_SIZES_EXTRA if "grid_size" in c]
    return configs + extra


## 1) tsallis_entropies_coarse


In [ ]:
all_results = {}
CURRENT_OUTPUT_DIR = OUTPUT_DIR / "tsallis_entropies_coarse"

state_entropy_configs = _add_grid_variants([
    dict(mode="spatial", family="tsallis", params=(0.5, 1.5, 2.0, 5.0), grid_size=10, smoothing=0.5),
    dict(mode="spatial", family="tsallis", params=(0.5, 1.5, 2.0, 5.0), escort_gamma=0.5, grid_size=10, smoothing=0.5),
    dict(mode="spatial", family="tsallis", params=(0.5, 1.5, 2.0, 5.0), escort_gamma=2.0, grid_size=10, smoothing=0.5),
])
all_results["StateDistributionEntropy"] = extract_feature_with_configs(
    StateDistributionEntropy, "StateDistributionEntropy", state_entropy_configs
)

duration_weighted_configs = _add_grid_variants([
    dict(mode="spatial", weights="duration", family="tsallis", params=(0.5, 1.5, 2.0, 5.0), grid_size=10, smoothing=0.5),
    dict(mode="spatial", weights="duration", family="tsallis", params=(0.5, 1.5, 2.0, 5.0), escort_gamma=0.5, grid_size=10, smoothing=0.5),
    dict(mode="spatial", weights="duration", family="tsallis", params=(0.5, 1.5, 2.0, 5.0), escort_gamma=2.0, grid_size=10, smoothing=0.5),
])
all_results["WeightedStateDistributionEntropy"] = extract_feature_with_configs(
    WeightedStateDistributionEntropy, "WeightedStateDistributionEntropy", duration_weighted_configs
)

edge_entropy_configs = _add_grid_variants([
    dict(mode="spatial", family="tsallis", params=(0.5, 1.5, 2.0, 5.0), grid_size=10, smoothing=0.5),
    dict(mode="spatial", family="tsallis", params=(0.5, 1.5, 2.0, 5.0), grid_size=10, smoothing=0.5, weights="duration"),
])
all_results["EdgeDistributionEntropy"] = extract_feature_with_configs(
    EdgeDistributionEntropy, "EdgeDistributionEntropy", edge_entropy_configs
)

saved = consolidate_and_save_features(all_results, output_dir=OUTPUT_DIR / "tsallis_entropies_coarse")
print(f"\nDone. Saved {len(saved)} consolidated files")

## 2) tsallis_entropies_fine


In [ ]:
FINE_PARAMS = (
    0.1, 0.25, 0.5, 0.75,
    1.0, 1.25, 1.5, 1.75,
    2.0, 2.5, 3.0, 4.0, 5.0
)

all_results_fine = {}
CURRENT_OUTPUT_DIR = OUTPUT_DIR / "tsallis_entropies_fine"

state_entropy_configs_fine = _add_grid_variants([
    dict(mode="spatial", family="tsallis", params=FINE_PARAMS, grid_size=10, smoothing=0.5),
    dict(mode="spatial", family="tsallis", params=FINE_PARAMS, escort_gamma=0.5, grid_size=10, smoothing=0.5),
    dict(mode="spatial", family="tsallis", params=FINE_PARAMS, escort_gamma=2.0, grid_size=10, smoothing=0.5),
])
all_results_fine["StateDistributionEntropy"] = extract_feature_with_configs(
    StateDistributionEntropy, "StateDistributionEntropy", state_entropy_configs_fine
)

duration_weighted_configs_fine = _add_grid_variants([
    dict(mode="spatial", weights="duration", family="tsallis", params=FINE_PARAMS, grid_size=10, smoothing=0.5),
    dict(mode="spatial", weights="duration", family="tsallis", params=FINE_PARAMS, escort_gamma=0.5, grid_size=10, smoothing=0.5),
    dict(mode="spatial", weights="duration", family="tsallis", params=FINE_PARAMS, escort_gamma=2.0, grid_size=10, smoothing=0.5),
])
all_results_fine["WeightedStateDistributionEntropy"] = extract_feature_with_configs(
    WeightedStateDistributionEntropy, "WeightedStateDistributionEntropy", duration_weighted_configs_fine
)

edge_entropy_configs_fine = _add_grid_variants([
    dict(mode="spatial", family="tsallis", params=FINE_PARAMS, grid_size=10, smoothing=0.5),
    dict(mode="spatial", family="tsallis", params=FINE_PARAMS, grid_size=10, smoothing=0.5, weights="duration"),
])
all_results_fine["EdgeDistributionEntropy"] = extract_feature_with_configs(
    EdgeDistributionEntropy, "EdgeDistributionEntropy", edge_entropy_configs_fine
)

saved_fine = consolidate_and_save_features(all_results_fine, output_dir=OUTPUT_DIR / "tsallis_entropies_fine")
print(f"\nDone. Saved {len(saved_fine)} consolidated files")

## 3) thermodynamic_features_coarse


In [ ]:
all_results = {}
CURRENT_OUTPUT_DIR = OUTPUT_DIR / "thermodynamic_features_coarse"

thermo_configs = _add_grid_variants([
    dict(energy="step_length", betas=(0.25, 0.5, 1.0, 2.0, 5.0), state_mode="spatial", grid_size=10, smoothing=0.5),
    dict(energy="turning", betas=(0.25, 0.5, 1.0, 2.0, 5.0), state_mode="spatial", grid_size=10, smoothing=0.5),
    dict(energy="surprisal_state", betas=(0.25, 0.5, 1.0, 2.0, 5.0), state_mode="spatial", grid_size=10, smoothing=0.5),
    dict(energy="surprisal_transition", betas=(0.25, 0.5, 1.0, 2.0, 5.0), state_mode="spatial", grid_size=10, smoothing=0.5),
    dict(energy="speed", betas=(0.25, 0.5, 1.0, 2.0, 5.0), state_mode="spatial", grid_size=10, smoothing=0.5),
    dict(energy="duration_inv", betas=(0.25, 0.5, 1.0, 2.0, 5.0), state_mode="spatial", grid_size=10, smoothing=0.5),
])

all_results["ThermodynamicObservables"] = extract_feature_with_configs(
    ThermodynamicObservables, "ThermodynamicObservables", thermo_configs
)

thermo_trans_configs = _add_grid_variants([
    dict(mode="spatial", cost="spatial", divergence="kl", alpha=2.0, betas=(0.5, 1, 1.5, 2.0, 5.0), grid_size=10, bounds="unit", smoothing=0.5, pi_mode="stationary"),
])
all_results["ThermodynamicTransitionDivergence"] = extract_feature_with_configs(
    ThermodynamicTransitionDivergence, "ThermodynamicTransitionDivergence", thermo_trans_configs
)

saved = consolidate_and_save_features(all_results, output_dir=OUTPUT_DIR / "thermodynamic_features_coarse")
print(f"\nDone. Saved {len(saved)} consolidated files")

## 4) thermodynamic_features_fine


In [ ]:
FINE_BETAS = (
    0.1, 0.25, 0.5, 0.75,
    1.0, 1.25, 1.5, 1.75,
    2.0, 2.5, 3.0, 4.0, 5.0
)

all_results_thermo_fine = {}
CURRENT_OUTPUT_DIR = OUTPUT_DIR / "thermodynamic_features_fine"


thermo2_configs_fine = _add_grid_variants([
    dict(energy="step_length", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=1.5, cumulative=False),
    dict(energy="turning", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=1.5, cumulative=False),
    dict(energy="surprisal_state", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=1.5, cumulative=False,
         state_mode="spatial", grid_size=10, smoothing=0.5),
    dict(energy="surprisal_transition", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=1.5, cumulative=False,
         state_mode="spatial", grid_size=10, smoothing=0.5),
    dict(energy="speed", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=1.5, cumulative=False),
    dict(energy="duration_inv", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=1.5, cumulative=False),
])

thermo3_configs_fine = _add_grid_variants([
    dict(energy="step_length", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=1, cumulative=False),
    dict(energy="turning", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=1, cumulative=False),
    dict(energy="surprisal_state", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=1, cumulative=False,
         state_mode="spatial", grid_size=10, smoothing=0.5),
    dict(energy="surprisal_transition", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=1, cumulative=False,
         state_mode="spatial", grid_size=10, smoothing=0.5),
    dict(energy="speed", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=1, cumulative=False),
    dict(energy="duration_inv", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=1, cumulative=False),
])

thermo4_configs_fine = _add_grid_variants([
    dict(energy="step_length", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=0.5, cumulative=False),
    dict(energy="turning", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=0.5, cumulative=False),
    dict(energy="surprisal_state", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=0.5, cumulative=False,
         state_mode="spatial", grid_size=10, smoothing=0.5),
    dict(energy="surprisal_transition", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=0.5, cumulative=False,
         state_mode="spatial", grid_size=10, smoothing=0.5),
    dict(energy="speed", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=0.5, cumulative=False),
    dict(energy="duration_inv", betas=FINE_BETAS, ensemble="tsallis", q_ensemble=0.5, cumulative=False),
])

all_results_thermo_fine["ThermodynamicObservablesExtended"] = extract_feature_with_configs(
    ThermodynamicObservablesExtended, "ThermodynamicObservablesExtended1", thermo2_configs_fine
)

all_results_thermo_fine["ThermodynamicObservablesExtended3"] = extract_feature_with_configs(
    ThermodynamicObservablesExtended, "ThermodynamicObservablesExtended2", thermo3_configs_fine
)

all_results_thermo_fine["ThermodynamicObservablesExtended4"] = extract_feature_with_configs(
    ThermodynamicObservablesExtended, "ThermodynamicObservablesExtended3", thermo4_configs_fine
)


thermo_trans_configs_fine = _add_grid_variants([
    dict(mode="spatial", cost="spatial", divergence="kl", alpha=2.0, betas=FINE_BETAS,
         grid_size=10, bounds="unit", smoothing=0.5, pi_mode="stationary"),
])
all_results_thermo_fine["ThermodynamicTransitionDivergence"] = extract_feature_with_configs(
    ThermodynamicTransitionDivergence, "ThermodynamicTransitionDivergence", thermo_trans_configs_fine
)

saved_thermo_fine = consolidate_and_save_features(all_results_thermo_fine, output_dir=OUTPUT_DIR / "thermodynamic_features_fine")
print(f"\nDone. Saved {len(saved_thermo_fine)} consolidated files (thermodynamic fine beta)")

## 5) shannon_entropies


In [ ]:
baseline_results = {}
CURRENT_OUTPUT_DIR = OUTPUT_DIR / "shannon_entropies"

state_entropy_baseline = [
    dict(mode="spatial", family="shannon", grid_size=10, smoothing=0.5),
]
baseline_results["StateDistributionEntropy"] = extract_feature_with_configs(
    StateDistributionEntropy, "StateDistributionEntropy_shannon", state_entropy_baseline
)

duration_weighted_baseline = [
    dict(mode="spatial", weights="duration", family="shannon", grid_size=10, smoothing=0.5),
]
baseline_results["WeightedStateDistributionEntropy"] = extract_feature_with_configs(
    WeightedStateDistributionEntropy, "WeightedStateDistributionEntropy_shannon", duration_weighted_baseline
)

edge_entropy_baseline = [
    dict(mode="spatial", family="shannon", grid_size=10, smoothing=0.5),
]
baseline_results["EdgeDistributionEntropy"] = extract_feature_with_configs(
    EdgeDistributionEntropy, "EdgeDistributionEntropy_shannon", edge_entropy_baseline
)

saved_baseline = consolidate_and_save_features(baseline_results, output_dir=OUTPUT_DIR / "shannon_entropies")
print(f"\nDone. Saved {len(saved_baseline)} consolidated files (Shannon baseline)")
